# Stage 05: Local Feature Extraction

**Status:** Implemented and unit-tested (`local_feature_extraction_model.py`,
`local_feature_extraction_dataset.py`, `pipeline/feature_extraction.py`). **Not trained, not
frozen.** This notebook stages the frozen upstream checkpoints (Stage 03 LWNet, Stage 04
Attention U-Net -- Experiment 2C), builds a small real-data demo input, constructs the Stage 05
model, and runs one safe sanity forward pass. It does **not** start real training -- see Section
9 (`RUN_TRAINING`).

## Objective

Extract a spatially resolved, segmentation-anchored local feature representation (Adaptive
Multi-Kernel CNN) for later consumption by Stage 07 (Adaptive Cross-Attention). See the approved
Stage 05 design document (reviewed and approved before implementation) for the full rationale.

## Expected Inputs

- Stage 02 processed RGB (3 channels)
- Stage 03 vessel probability map (1 channel)
- Stage 04 four lesion probability maps (4 channels)

Concatenated into a single `(512, 512, 8)` tensor -- channels 0-2 RGB, channel 3 vessel
probability, channels 4-7 lesion probabilities (Microaneurysm, Haemorrhage, HardExudate,
SoftExudate). Never binary masks, never ground-truth masks.

## Expected Outputs

- `local_features`, shape `(32, 32, 256)` -- spatially resolved, never globally pooled -- for
  Stage 07 (Adaptive Cross-Attention).

## Datasets

APTOS 2019 only (3662 labeled images, `diagnosis` 0-4). IDRiD is **not** used to train Stage 05
directly -- it is consumed only indirectly, through the frozen Stage 04 checkpoint this notebook
loads. EyeQ has no role here.

## Dependencies

Stage 02 (Image Preprocessing), Stage 03 (Vessel Segmentation -- frozen, pretrained LWNet),
Stage 04 (Lesion Segmentation -- frozen, Experiment 2C, Mean Dice 0.1314 / Mean IoU 0.0766 on
the official IDRiD test set).

## Training Status

Stage 05 has **no standalone training procedure** -- it has no independent local-feature ground
truth of its own. It will eventually be trained jointly with Stage 06 (Global Feature
Extraction), Stage 07 (Adaptive Cross-Attention), RACAF, and Stage 08 (CORN), through the
downstream CORN ordinal loss, once those stages exist. That joint training script is not
implemented yet, and is out of scope for this notebook.

---

This notebook does not modify Stages 01-04 or RACAF, does not implement Stage 06/07/RACAF/CORN,
and does not start real training.

### Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s module docstring for why this is intentionally duplicated.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

### Imports

The reusable `colab/common/` infrastructure works the same way it does for every other stage
notebook. The stage-specific pieces below (`local_feature_extraction_model.py`,
`local_feature_extraction_dataset.py`) are now implemented -- imported in Section 4 onward, once
the frozen Stage 03/04 checkpoints they depend on are staged.

In [ ]:
import setup

setup_info = setup.setup()

import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)


## 3. Configuration

Stage 05's fixed contract (input/output shapes, channel semantics) plus this notebook's own demo
scope. `IMAGE_SIZE`/`STAGE_FILTERS` come directly from `local_feature_extraction_model.py`'s own
defaults -- not re-declared or overridden here, so this notebook can never silently drift from
the approved design's `(512, 512, 8) -> (32, 32, 256)` contract.

In [ ]:
import numpy as np

import config
import local_feature_extraction_dataset as lfed
import local_feature_extraction_model as lfem

IMAGE_SIZE = lfem.DEFAULT_INPUT_SHAPE[:2]          # (512, 512)
NUM_CHANNELS = lfed.NUM_CHANNELS                    # 8 (3 RGB + 1 vessel + 4 lesion)
STAGE_FILTERS = lfem.DEFAULT_STAGE_FILTERS          # (32, 64, 128, 256)
OUTPUT_SPATIAL_SIZE = lfem.OUTPUT_SPATIAL_SIZE       # 32
OUTPUT_CHANNELS = lfem.OUTPUT_CHANNELS               # 256

# This notebook only ever builds a SMALL real-data demo subset -- staging and
# processing the full 3662-image labeled APTOS set through Stage 03/04 inference
# is the future joint Stage 05-08 training script's job, not this sanity-check
# notebook's. See Section 5.
DEMO_SAMPLE_COUNT = 6

print(f"Stage 05 input shape:  (B, {IMAGE_SIZE[0]}, {IMAGE_SIZE[1]}, {NUM_CHANNELS})")
print(f"Stage 05 output shape: (B, {OUTPUT_SPATIAL_SIZE}, {OUTPUT_SPATIAL_SIZE}, {OUTPUT_CHANNELS})")
print(f"Multi-kernel block filter progression: {STAGE_FILTERS}")
print(f"Demo sample count for this notebook: {DEMO_SAMPLE_COUNT} (of 3662 labeled APTOS images total)")


## 4. Frozen Stage 03 / Stage 04 Checkpoint Staging + Loading

Copies the vendored Stage 03 LWNet checkpoint (`best_model.pth`, `config.cfg`) and the frozen
Stage 04 Experiment 2C checkpoint (`best_model.keras`) from their **exported_models** Drive
locations -- identical staging pattern to `stage04_lesion_segmentation.ipynb`'s own Section 4,
extended here to also stage Stage 04's exported checkpoint. This notebook never reads, moves, or
modifies `experiments/LesionSegmentation/2026-08-24_11-54-34/` (the timestamped training-run
folder) -- only the separate, permanent `exported_models/LesionSegmentation/best_model.keras`
artifact that run already produced. Neither checkpoint's weights are ever updated by anything in
this notebook.

**One-time manual prerequisite:** both checkpoints must already be uploaded to their respective
`MyDrive/DiabeticRetinopathy/exported_models/<Module>/` folders (Stage 03's, per
`stage03_vessel_segmentation.ipynb`; Stage 04's, per `stage04_lesion_segmentation.ipynb` Section
13's export step).

In [ ]:
import shutil

from lesion_segmentation_model import DEFAULT_MODEL_PATH as DEFAULT_LESION_MODEL_PATH
from lesion_segmentation_model import load_lesion_model
from vessel_segmentation_inference import DEFAULT_MODEL_PATH as DEFAULT_VESSEL_MODEL_PATH
from vessel_segmentation_inference import load_vessel_model

# --- Stage 03 (vessel, frozen, pretrained LWNet) ---
VESSEL_SEG_DRIVE_DIR = colab_config.DRIVE.exported_model_dir("VesselSegmentation")
VESSEL_CHECKPOINT_FILES = ("best_model.pth", "config.cfg")

os.makedirs(config.VESSEL_SEG_MODEL_DIR, exist_ok=True)
for filename in VESSEL_CHECKPOINT_FILES:
    src = os.path.join(VESSEL_SEG_DRIVE_DIR, filename)
    dst = os.path.join(config.VESSEL_SEG_MODEL_DIR, filename)
    if not os.path.isfile(src):
        raise RuntimeError(
            f"Vessel Segmentation artifact not found on Drive: {src}. Upload the vendored "
            "LWNet checkpoint there first -- see stage04_lesion_segmentation.ipynb Section 4."
        )
    shutil.copy2(src, dst)
    print(f"Staged {src} -> {dst} ({os.path.getsize(dst):,} bytes)")

vessel_model = load_vessel_model(DEFAULT_VESSEL_MODEL_PATH)
n_vessel_params = sum(p.numel() for p in vessel_model.parameters())
print(f"\nLoaded frozen Stage 03 checkpoint ({n_vessel_params:,} params)")

# --- Stage 04 (lesion, frozen, Experiment 2C -- Weighted-Pooled Dice) ---
LESION_SEG_DRIVE_DIR = colab_config.DRIVE.exported_model_dir("LesionSegmentation")
LESION_CHECKPOINT_FILE = "best_model.keras"

os.makedirs(config.LESION_SEG_MODEL_DIR, exist_ok=True)
lesion_src = os.path.join(LESION_SEG_DRIVE_DIR, LESION_CHECKPOINT_FILE)
lesion_dst = os.path.join(config.LESION_SEG_MODEL_DIR, LESION_CHECKPOINT_FILE)
if not os.path.isfile(lesion_src):
    raise RuntimeError(
        f"Lesion Segmentation (Experiment 2C) checkpoint not found on Drive: {lesion_src}. "
        "Export the frozen best_model.keras there first -- see "
        "stage04_lesion_segmentation.ipynb Section 13. This notebook never trains Stage 04."
    )
shutil.copy2(lesion_src, lesion_dst)
print(f"Staged {lesion_src} -> {lesion_dst} ({os.path.getsize(lesion_dst):,} bytes)")

lesion_model = load_lesion_model(DEFAULT_LESION_MODEL_PATH)
print(f"Loaded frozen Stage 04 checkpoint ({lesion_model.count_params():,} params, "
      "Experiment 2C -- Weighted-Pooled Dice, Mean Dice 0.1314 / Mean IoU 0.0766 on the "
      "official IDRiD test set)")


## 5. APTOS 2019 Demo Subset Staging

Stages the full `train.csv` (small) plus only `DEMO_SAMPLE_COUNT` images (of 3662 total labeled
images) from Drive -- enough for a real, non-synthetic sanity forward pass without this notebook
performing thousands of Stage 03/04 inference calls. Building the full labeled dataset is the
future joint Stage 05-08 training script's job (`local_feature_extraction_dataset.
load_local_feature_extraction_datasets` already implements it and works unchanged against the
full dataset once staged), not this notebook's.

In [ ]:
import csv
import posixpath

APTOS_TRAIN_CSV_DRIVE = posixpath.join(colab_config.APTOS2019_DATASET_DIR, "raw", "train.csv")
APTOS_TRAIN_IMAGES_DRIVE = posixpath.join(colab_config.APTOS2019_DATASET_DIR, "raw", "train_images")

os.makedirs(os.path.dirname(lfed.DEFAULT_TRAIN_CSV), exist_ok=True)
os.makedirs(lfed.DEFAULT_TRAIN_IMAGE_DIR, exist_ok=True)

if not os.path.isfile(APTOS_TRAIN_CSV_DRIVE):
    raise RuntimeError(f"APTOS2019 train.csv not found on Drive: {APTOS_TRAIN_CSV_DRIVE}.")
shutil.copy2(APTOS_TRAIN_CSV_DRIVE, lfed.DEFAULT_TRAIN_CSV)

with open(lfed.DEFAULT_TRAIN_CSV, newline="", encoding="utf-8") as f:
    demo_rows = list(csv.DictReader(f))[:DEMO_SAMPLE_COUNT]

for row in demo_rows:
    filename = f"{row['id_code']}.png"
    src = posixpath.join(APTOS_TRAIN_IMAGES_DRIVE, filename)
    dst = os.path.join(lfed.DEFAULT_TRAIN_IMAGE_DIR, filename)
    if not os.path.isfile(src):
        raise RuntimeError(f"APTOS2019 image not found on Drive: {src}.")
    shutil.copy2(src, dst)

print(f"Staged train.csv (full, {sum(1 for _ in open(lfed.DEFAULT_TRAIN_CSV, encoding='utf-8')) - 1} "
      f"labeled rows) + {len(demo_rows)} demo images")
print(f"Demo id_codes / diagnoses: {[(r['id_code'], r['diagnosis']) for r in demo_rows]}")


## 6. Stage 05 Input Construction

Builds one real `(512, 512, 8)` Stage 05 input tensor per demo image, via
`local_feature_extraction_dataset._build_sample` -- Stage 02 preprocessing applied live, Stage
03/04 outputs computed via their frozen, already-loaded models (cached to disk afterward, so a
re-run of this cell does not re-run inference). No ground-truth mask of any kind is read for
these images -- APTOS has none to begin with.

In [ ]:
demo_inputs = []
demo_labels = []
for row in demo_rows:
    id_code = row["id_code"]
    diagnosis = int(row["diagnosis"])
    x, y = lfed._build_sample(
        id_code, diagnosis, lfed.DEFAULT_TRAIN_IMAGE_DIR, lfed.DEFAULT_CACHE_DIR,
        vessel_model, lesion_model, image_size=IMAGE_SIZE,
    )
    demo_inputs.append(x)
    demo_labels.append(y)

demo_batch = np.stack(demo_inputs, axis=0)
print(f"Demo batch shape: {demo_batch.shape}  (dtype={demo_batch.dtype})")
print(f"Channel 0-2 (RGB) range:    [{demo_batch[..., 0:3].min():.3f}, {demo_batch[..., 0:3].max():.3f}]")
print(f"Channel 3 (vessel) range:   [{demo_batch[..., 3].min():.3f}, {demo_batch[..., 3].max():.3f}]")
print(f"Channels 4-7 (lesion) range: [{demo_batch[..., 4:8].min():.3f}, {demo_batch[..., 4:8].max():.3f}]")
print(f"Demo labels (APTOS DR grade 0-4): {demo_labels}")


## 7. Stage 05 Model Construction

Builds the Adaptive Multi-Kernel CNN via `build_local_feature_extractor()` -- random
initialization, no ImageNet weights (see `local_feature_extraction_model.py`'s docstring for
why), **uncompiled** (Stage 05 has no standalone loss). Not trained.

In [ ]:
local_feature_model = lfem.build_local_feature_extractor()
local_feature_model.summary()

total_params = local_feature_model.count_params()
trainable_params = int(sum(np.prod(v.shape) for v in local_feature_model.trainable_variables))
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Compiled with a loss: {local_feature_model.loss is not None} (must be False -- see Section 7's markdown)")


## 8. Sanity Forward Pass

Runs the real demo batch from Section 6 through the freshly built, untrained Stage 05 model.
This verifies the `(512, 512, 8) -> (32, 32, 256)` contract end-to-end on real (not synthetic)
data -- it is not a training step, and the resulting features are meaningless (random weights),
which is expected and fine for a plumbing check.

In [ ]:
demo_output = local_feature_model.predict(demo_batch, verbose=0)

print(f"Input shape:  {demo_batch.shape}")
print(f"Output shape: {demo_output.shape}")
print(f"Output is finite: {np.isfinite(demo_output).all()}")

expected_output_shape = (demo_batch.shape[0], OUTPUT_SPATIAL_SIZE, OUTPUT_SPATIAL_SIZE, OUTPUT_CHANNELS)
assert demo_output.shape == expected_output_shape, (
    f"Expected {expected_output_shape}, got {demo_output.shape}"
)
print(f"\nMatches the approved output contract: (B, {OUTPUT_SPATIAL_SIZE}, {OUTPUT_SPATIAL_SIZE}, {OUTPUT_CHANNELS})")


## 9. Training Status

**`RUN_TRAINING` stays `False` in this notebook, following the same project-standard
"prepared, not auto-started" convention every other stage notebook uses.** Unlike Stage 04's
notebook, there is nothing this flag could start even if set to `True` -- Stage 05 has no
standalone training procedure (`LocalFeatureExtractionStage.train()` raises
`NotImplementedError` by design, see `local_feature_extraction_model.py`'s docstring). Real
training will happen in a future, not-yet-implemented joint Stage 05-08 + RACAF training script,
once Stages 06 and 07 exist.

In [ ]:
RUN_TRAINING = False  # Stage 05 has no standalone training procedure -- see this section's markdown.

if not RUN_TRAINING:
    print("RUN_TRAINING is False -- no training was started in this notebook.")
    print("Stage 05 is IMPLEMENTED and TESTED, but NOT TRAINED and NOT FROZEN.")
else:
    raise RuntimeError(
        "Stage 05 has no standalone training procedure to run -- see "
        "LocalFeatureExtractionStage.train()'s NotImplementedError. Real training happens only "
        "as part of the future joint Stage 05-08 + RACAF training script."
    )


## 10. Summary

In [ ]:
print("=" * 72)
print("Stage 05: Local Feature Extraction -- Summary")
print("=" * 72)
print(f"Architecture: Adaptive Multi-Kernel CNN, {len(STAGE_FILTERS)} multi-kernel blocks, "
      f"filters {STAGE_FILTERS}")
print(f"Input:  (B, {IMAGE_SIZE[0]}, {IMAGE_SIZE[1]}, {NUM_CHANNELS}) -- Stage 02 RGB + "
      "Stage 03 vessel probability + Stage 04 4 lesion probabilities")
print(f"Output: (B, {OUTPUT_SPATIAL_SIZE}, {OUTPUT_SPATIAL_SIZE}, {OUTPUT_CHANNELS}) -- spatially resolved, not pooled")
print(f"Total parameters: {total_params:,} (trainable: {trainable_params:,})")
print(f"Demo forward pass: input {demo_batch.shape} -> output {demo_output.shape}")
print(f"\nFrozen dependencies used (unmodified): Stage 02 preprocessing, Stage 03 LWNet "
      f"({n_vessel_params:,} params), Stage 04 Attention U-Net Experiment 2C "
      f"({lesion_model.count_params():,} params, Mean Dice 0.1314 / Mean IoU 0.0766)")
print(f"\nSTATUS: IMPLEMENTED, TESTED, NOT TRAINED, NOT FROZEN.")
print("Training will be a separate step, after Stages 06/07 exist and this implementation is reviewed.")
